# ACDADA — Notebook 02: Threat Detection Agent

**Agent 1: Binary Intrusion Detection (Malicious vs Benign)**

This notebook implements:
1. CNN-based network traffic classifier
2. LSTM-based sequential traffic classifier
3. Hybrid CNN-LSTM model
4. Training with early stopping, LR scheduling, class weighting
5. Comprehensive evaluation (ROC-AUC, PR-AUC, Confusion Matrix)
6. Model export for deployment

In [ ]:
# ============================================================
# DATASET LINKS (processed data from Notebook 01)
# ============================================================
#
# Original raw datasets:
# 1. CIC-IDS-2017: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
# 2. UNSW-NB15:    https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
# 3. Bot-IoT:      https://www.kaggle.com/datasets/vigneshvenkateswaran/bot-iot-dataset
# 4. BETH:         https://www.kaggle.com/datasets/katehighnam/beth-dataset
#
# Processed data expected at: ../data/processed/<dataset>/
# Run Notebook 01 first to generate processed splits.
# ============================================================

## 1. Imports & Configuration

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, roc_curve,
    f1_score, accuracy_score, matthews_corrcoef
)

import joblib
import json
import gc

warnings.filterwarnings('ignore')

# Paths
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
LOGS_DIR = Path('../logs')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Hyperparameters
RANDOM_STATE = 42
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
EPOCHS = 100
PATIENCE = 10

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_STATE)

## 2. Load Processed Data

In [ ]:
def load_dataset_splits(dataset_name: str, processed_dir: Path) -> dict:
    """Load preprocessed train/val/test splits from disk."""
    dataset_dir = processed_dir / dataset_name
    
    if not dataset_dir.exists():
        print(f'[WARNING] {dataset_dir} does not exist. Run Notebook 01 first.')
        return None
    
    data = {}
    for key in ['X_train', 'X_val', 'X_test',
                'y_train_binary', 'y_val_binary', 'y_test_binary']:
        filepath = dataset_dir / f'{key}.npy'
        if filepath.exists():
            data[key] = np.load(filepath)
            print(f'  Loaded {key}: {data[key].shape} ({data[key].dtype})')
    
    # Feature names
    fn_path = dataset_dir / 'feature_names.npy'
    if fn_path.exists():
        data['feature_names'] = np.load(fn_path, allow_pickle=True).tolist()
    
    return data

# Load primary dataset (CICIDS2017 or unified)
DATASET_NAME = 'cicids2017'  # Change to 'unified' for cross-dataset training

# Try unified first, fallback to individual
for name in ['unified', 'cicids2017', 'unsw_nb15', 'bot_iot']:
    data_path = PROCESSED_DIR / name
    if data_path.exists():
        DATASET_NAME = name
        break

print(f'\nLoading dataset: {DATASET_NAME}')
data = load_dataset_splits(DATASET_NAME, PROCESSED_DIR)

if data is None:
    raise FileNotFoundError('No processed data found. Run Notebook 01 first.')

X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
y_train = data['y_train_binary']
y_val = data['y_val_binary']
y_test = data['y_test_binary']

N_FEATURES = X_train.shape[1]
print(f'\nDataset: {DATASET_NAME}')
print(f'Features: {N_FEATURES}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Train class dist: {np.bincount(y_train)}')
print(f'Attack ratio: {y_train.mean():.3f}')

## 3. Data Loaders with Class Weighting

In [ ]:
def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test,
                       batch_size=512, use_weighted_sampler=True):
    """
    Create PyTorch DataLoaders with optional weighted random sampling
    to handle class imbalance.
    """
    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.LongTensor(y_train)
    X_val_t = torch.FloatTensor(X_val)
    y_val_t = torch.LongTensor(y_val)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.LongTensor(y_test)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    
    # Weighted sampler for class imbalance
    sampler = None
    shuffle = True
    if use_weighted_sampler:
        class_counts = np.bincount(y_train)
        class_weights = 1.0 / class_counts
        sample_weights = class_weights[y_train]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(y_train),
            replacement=True
        )
        shuffle = False  # Sampler handles shuffling
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=shuffle, sampler=sampler,
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size * 2,
                            shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size * 2,
                             shuffle=False, num_workers=0, pin_memory=True)
    
    # Compute class weights for loss function
    class_counts = np.bincount(y_train)
    total = len(y_train)
    class_weight_tensor = torch.FloatTensor(
        [total / (2 * c) for c in class_counts]
    ).to(DEVICE)
    
    print(f'DataLoaders created:')
    print(f'  Train batches: {len(train_loader)}')
    print(f'  Val batches: {len(val_loader)}')
    print(f'  Test batches: {len(test_loader)}')
    print(f'  Class weights: {class_weight_tensor.cpu().numpy()}')
    
    return train_loader, val_loader, test_loader, class_weight_tensor

train_loader, val_loader, test_loader, class_weights = create_dataloaders(
    X_train, y_train, X_val, y_val, X_test, y_test,
    batch_size=BATCH_SIZE
)

---
## 4. Model Architectures

### 4.1 — 1D-CNN Threat Detector

In [ ]:
class CNNThreatDetector(nn.Module):
    """
    1D Convolutional Neural Network for binary intrusion detection.
    Treats each network flow's features as a 1D signal.
    
    Architecture:
    - 3 Conv1D blocks with BatchNorm, ReLU, Dropout
    - Global Average Pooling
    - 2 Fully Connected layers
    - Output: 2 classes (Benign / Attack)
    """
    
    def __init__(self, n_features: int, dropout: float = 0.3):
        super().__init__()
        self.n_features = n_features
        
        # Conv blocks — input shape: (batch, 1, n_features)
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """Kaiming initialization for conv and linear layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # x shape: (batch, n_features) → (batch, 1, n_features)
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x).squeeze(-1)  # (batch, 256)
        x = self.classifier(x)
        return x

# Test
model_cnn = CNNThreatDetector(N_FEATURES).to(DEVICE)
dummy = torch.randn(4, N_FEATURES).to(DEVICE)
out = model_cnn(dummy)
print(f'CNN model output shape: {out.shape}')
print(f'CNN parameters: {sum(p.numel() for p in model_cnn.parameters()):,}')

### 4.2 — LSTM Threat Detector

In [ ]:
class LSTMThreatDetector(nn.Module):
    """
    Bidirectional LSTM for binary intrusion detection.
    Treats features as a sequence with temporal patterns.
    
    Architecture:
    - Feature projection layer
    - 2-layer Bidirectional LSTM
    - Attention mechanism
    - FC classifier
    """
    
    def __init__(self, n_features: int, hidden_size: int = 128,
                 n_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        
        # Project features into sequence chunks
        self.seq_len = max(n_features // 8, 4)  # Create a pseudo-sequence
        self.input_size = (n_features + self.seq_len - 1) // self.seq_len
        self.padded_size = self.seq_len * self.input_size
        
        # Projection
        self.projection = nn.Sequential(
            nn.Linear(n_features, self.padded_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
        )
        
        # LSTM
        self.lstm = nn.LSTM(
            input_size=self.input_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0,
        )
        
        # Attention
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        
        # Project and reshape into sequence
        x = self.projection(x)  # (batch, padded_size)
        x = x.view(batch_size, self.seq_len, self.input_size)  # (batch, seq_len, input_size)
        
        # LSTM
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden*2)
        
        # Attention
        attn_weights = self.attention(lstm_out)  # (batch, seq_len, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        context = torch.sum(lstm_out * attn_weights, dim=1)  # (batch, hidden*2)
        
        # Classify
        out = self.classifier(context)
        return out

# Test
model_lstm = LSTMThreatDetector(N_FEATURES).to(DEVICE)
out = model_lstm(dummy)
print(f'LSTM model output shape: {out.shape}')
print(f'LSTM parameters: {sum(p.numel() for p in model_lstm.parameters()):,}')

### 4.3 — Hybrid CNN-LSTM Threat Detector

In [ ]:
class HybridCNNLSTMDetector(nn.Module):
    """
    Hybrid CNN-LSTM for binary intrusion detection.
    CNN extracts local patterns, LSTM captures sequential dependencies.
    
    Architecture:
    - 2 Conv1D blocks for feature extraction
    - Bidirectional LSTM on CNN output
    - Attention pooling
    - FC classifier with residual connection
    """
    
    def __init__(self, n_features: int, cnn_channels: int = 64,
                 lstm_hidden: int = 128, dropout: float = 0.3):
        super().__init__()
        self.n_features = n_features
        
        # CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(cnn_channels, cnn_channels * 2, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        
        # LSTM on CNN output — treating spatial as temporal
        self.lstm = nn.LSTM(
            input_size=cnn_channels * 2,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        
        # Attention
        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            nn.Tanh(),
            nn.Linear(lstm_hidden, 1),
        )
        
        # Residual from CNN (global avg pool)
        self.cnn_pool = nn.AdaptiveAvgPool1d(1)
        
        # Final classifier
        combined_size = lstm_hidden * 2 + cnn_channels * 2
        self.classifier = nn.Sequential(
            nn.Linear(combined_size, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 2),
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # CNN path
        x_cnn = x.unsqueeze(1)  # (batch, 1, features)
        cnn_out = self.cnn(x_cnn)  # (batch, channels*2, features)
        
        # CNN residual
        cnn_pooled = self.cnn_pool(cnn_out).squeeze(-1)  # (batch, channels*2)
        
        # LSTM path — transpose for seq input
        lstm_input = cnn_out.permute(0, 2, 1)  # (batch, features, channels*2)
        lstm_out, _ = self.lstm(lstm_input)  # (batch, features, hidden*2)
        
        # Attention
        attn_weights = self.attention(lstm_out)  # (batch, features, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        lstm_context = torch.sum(lstm_out * attn_weights, dim=1)  # (batch, hidden*2)
        
        # Combine CNN + LSTM
        combined = torch.cat([lstm_context, cnn_pooled], dim=1)
        out = self.classifier(combined)
        return out

# Test
model_hybrid = HybridCNNLSTMDetector(N_FEATURES).to(DEVICE)
out = model_hybrid(dummy)
print(f'Hybrid CNN-LSTM output shape: {out.shape}')
print(f'Hybrid parameters: {sum(p.numel() for p in model_hybrid.parameters()):,}')

---
## 5. Training Engine

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    
    def __init__(self, patience: int = 10, min_delta: float = 1e-4,
                 mode: str = 'min', verbose: bool = True):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.should_stop = False
        self.best_model_state = None
    
    def __call__(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return
        
        if self.mode == 'min':
            improved = score < self.best_score - self.min_delta
        else:
            improved = score > self.best_score + self.min_delta
        
        if improved:
            self.best_score = score
            self.counter = 0
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.verbose:
                print(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.should_stop = True


class ThreatDetectionTrainer:
    """
    Training engine for threat detection models.
    Supports: weighted loss, LR scheduling, early stopping,
    gradient clipping, mixed precision (AMP).
    """
    
    def __init__(self, model, class_weights=None, lr=1e-3, device='cpu'):
        self.model = model.to(device)
        self.device = device
        
        # Loss with class weighting
        self.criterion = nn.CrossEntropyLoss(
            weight=class_weights if class_weights is not None else None
        )
        
        # Optimizer
        self.optimizer = optim.AdamW(
            model.parameters(), lr=lr, weight_decay=1e-4
        )
        
        # LR Scheduler
        self.scheduler = ReduceLROnPlateau(
            self.optimizer, mode='min', patience=5,
            factor=0.5, min_lr=1e-6, verbose=True
        )
        
        # Mixed precision
        self.scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
        
        # History
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'val_f1': [], 'val_auc': [], 'lr': []
        }
    
    def train_epoch(self, train_loader):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)
            
            self.optimizer.zero_grad()
            
            if self.scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = self.model(X_batch)
                    loss = self.criterion(outputs, y_batch)
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(X_batch)
                loss = self.criterion(outputs, y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            
            total_loss += loss.item() * X_batch.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == y_batch).sum().item()
            total += X_batch.size(0)
        
        return total_loss / total, correct / total
    
    @torch.no_grad()
    def validate(self, val_loader):
        """Validate the model."""
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_probs = []
        all_labels = []
        
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)
            
            outputs = self.model(X_batch)
            loss = self.criterion(outputs, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
        
        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)
        all_labels = np.array(all_labels)
        
        avg_loss = total_loss / len(all_labels)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='binary')
        
        try:
            auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            auc = 0.0
        
        return avg_loss, acc, f1, auc, all_preds, all_probs, all_labels
    
    def fit(self, train_loader, val_loader, epochs=100, patience=10):
        """Full training loop with early stopping."""
        early_stopping = EarlyStopping(patience=patience, mode='min')
        
        print(f'\nTraining {self.model.__class__.__name__}...')
        print(f'  Epochs: {epochs} | Patience: {patience} | LR: {self.optimizer.param_groups[0]["lr"]}')
        print(f'  Device: {self.device}')
        print('-' * 80)
        
        for epoch in range(1, epochs + 1):
            # Train
            train_loss, train_acc = self.train_epoch(train_loader)
            
            # Validate
            val_loss, val_acc, val_f1, val_auc, _, _, _ = self.validate(val_loader)
            
            # LR Scheduling
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # History
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            self.history['val_f1'].append(val_f1)
            self.history['val_auc'].append(val_auc)
            self.history['lr'].append(current_lr)
            
            # Print progress
            if epoch % 5 == 0 or epoch == 1:
                print(f'  Epoch {epoch:3d}/{epochs} | '
                      f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
                      f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f} AUC: {val_auc:.4f} | '
                      f'LR: {current_lr:.2e}')
            
            # Early stopping
            early_stopping(val_loss, self.model)
            if early_stopping.should_stop:
                print(f'\n  Early stopping at epoch {epoch}')
                break
        
        # Restore best model
        if early_stopping.best_model_state is not None:
            self.model.load_state_dict(early_stopping.best_model_state)
            print(f'  Restored best model (val_loss: {early_stopping.best_score:.4f})')
        
        return self.history

---
## 6. Train All Models

In [ ]:
# ============================================================
# Train CNN Model
# ============================================================
print('\n' + '='*60)
print('  TRAINING: CNN Threat Detector')
print('='*60)

model_cnn = CNNThreatDetector(N_FEATURES, dropout=0.3).to(DEVICE)
trainer_cnn = ThreatDetectionTrainer(
    model_cnn, class_weights=class_weights,
    lr=LEARNING_RATE, device=DEVICE
)
history_cnn = trainer_cnn.fit(
    train_loader, val_loader,
    epochs=EPOCHS, patience=PATIENCE
)

In [ ]:
# ============================================================
# Train LSTM Model
# ============================================================
print('\n' + '='*60)
print('  TRAINING: LSTM Threat Detector')
print('='*60)

model_lstm = LSTMThreatDetector(N_FEATURES, hidden_size=128, dropout=0.3).to(DEVICE)
trainer_lstm = ThreatDetectionTrainer(
    model_lstm, class_weights=class_weights,
    lr=LEARNING_RATE, device=DEVICE
)
history_lstm = trainer_lstm.fit(
    train_loader, val_loader,
    epochs=EPOCHS, patience=PATIENCE
)

In [ ]:
# ============================================================
# Train Hybrid CNN-LSTM Model
# ============================================================
print('\n' + '='*60)
print('  TRAINING: Hybrid CNN-LSTM Threat Detector')
print('='*60)

model_hybrid = HybridCNNLSTMDetector(N_FEATURES, dropout=0.3).to(DEVICE)
trainer_hybrid = ThreatDetectionTrainer(
    model_hybrid, class_weights=class_weights,
    lr=LEARNING_RATE, device=DEVICE
)
history_hybrid = trainer_hybrid.fit(
    train_loader, val_loader,
    epochs=EPOCHS, patience=PATIENCE
)

---
## 7. Training Visualization

In [ ]:
def plot_training_history(histories: dict):
    """Plot training curves for all models."""
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle('Training History — Threat Detection Models', fontsize=16, fontweight='bold')
    
    colors = {'CNN': '#2196F3', 'LSTM': '#FF9800', 'Hybrid': '#4CAF50'}
    
    for name, history in histories.items():
        color = colors.get(name, 'gray')
        epochs_range = range(1, len(history['train_loss']) + 1)
        
        # Loss
        axes[0, 0].plot(epochs_range, history['train_loss'], '--', color=color, alpha=0.5, label=f'{name} train')
        axes[0, 0].plot(epochs_range, history['val_loss'], '-', color=color, label=f'{name} val')
        
        # Accuracy
        axes[0, 1].plot(epochs_range, history['train_acc'], '--', color=color, alpha=0.5, label=f'{name} train')
        axes[0, 1].plot(epochs_range, history['val_acc'], '-', color=color, label=f'{name} val')
        
        # F1 Score
        axes[0, 2].plot(epochs_range, history['val_f1'], '-', color=color, label=f'{name}')
        
        # AUC
        axes[1, 0].plot(epochs_range, history['val_auc'], '-', color=color, label=f'{name}')
        
        # LR
        axes[1, 1].plot(epochs_range, history['lr'], '-', color=color, label=f'{name}')
    
    axes[0, 0].set_title('Loss'); axes[0, 0].set_ylabel('Loss'); axes[0, 0].legend()
    axes[0, 1].set_title('Accuracy'); axes[0, 1].set_ylabel('Accuracy'); axes[0, 1].legend()
    axes[0, 2].set_title('Validation F1'); axes[0, 2].set_ylabel('F1 Score'); axes[0, 2].legend()
    axes[1, 0].set_title('Validation AUC-ROC'); axes[1, 0].set_ylabel('AUC'); axes[1, 0].legend()
    axes[1, 1].set_title('Learning Rate'); axes[1, 1].set_ylabel('LR'); axes[1, 1].set_yscale('log'); axes[1, 1].legend()
    
    # Summary table
    axes[1, 2].axis('off')
    summary_data = []
    for name, history in histories.items():
        summary_data.append([
            name,
            f"{min(history['val_loss']):.4f}",
            f"{max(history['val_acc']):.4f}",
            f"{max(history['val_f1']):.4f}",
            f"{max(history['val_auc']):.4f}",
        ])
    table = axes[1, 2].table(
        cellText=summary_data,
        colLabels=['Model', 'Best Loss', 'Best Acc', 'Best F1', 'Best AUC'],
        loc='center', cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.5)
    axes[1, 2].set_title('Summary', fontsize=12)
    
    for ax in axes.flatten()[:5]:
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history({
    'CNN': history_cnn,
    'LSTM': history_lstm,
    'Hybrid': history_hybrid,
})

---
## 8. Test Set Evaluation

In [ ]:
def evaluate_model(trainer, test_loader, model_name: str):
    """Comprehensive evaluation on test set."""
    print(f'\n{"="*60}')
    print(f'  TEST EVALUATION: {model_name}')
    print(f'{"="*60}')
    
    _, test_acc, test_f1, test_auc, preds, probs, labels = trainer.validate(test_loader)
    mcc = matthews_corrcoef(labels, preds)
    
    print(f'\n  Accuracy:  {test_acc:.4f}')
    print(f'  F1 Score:  {test_f1:.4f}')
    print(f'  AUC-ROC:   {test_auc:.4f}')
    print(f'  MCC:       {mcc:.4f}')
    
    # Classification report
    print(f'\n  Classification Report:')
    print(classification_report(labels, preds, target_names=['Benign', 'Attack']))
    
    # Confusion Matrix
    cm = confusion_matrix(labels, preds)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{model_name} — Test Set Evaluation', fontsize=14, fontweight='bold')
    
    # Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
    axes[0].set_title('Confusion Matrix')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {test_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    axes[2].plot(recall, precision, 'g-', linewidth=2, label=f'AP = {ap:.4f}')
    axes[2].set_title('Precision-Recall Curve'); axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'model': model_name, 'accuracy': test_acc, 'f1': test_f1,
        'auc': test_auc, 'mcc': mcc, 'ap': ap,
        'predictions': preds, 'probabilities': probs, 'labels': labels
    }

# Evaluate all models
results = {}
results['CNN'] = evaluate_model(trainer_cnn, test_loader, 'CNN Threat Detector')
results['LSTM'] = evaluate_model(trainer_lstm, test_loader, 'LSTM Threat Detector')
results['Hybrid'] = evaluate_model(trainer_hybrid, test_loader, 'Hybrid CNN-LSTM Detector')

In [ ]:
# Model comparison summary
print('\n' + '='*60)
print('  MODEL COMPARISON SUMMARY')
print('='*60)

comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': r['accuracy'],
        'F1 Score': r['f1'],
        'AUC-ROC': r['auc'],
        'MCC': r['mcc'],
        'Avg Precision': r['ap'],
    }
    for name, r in results.items()
])

comparison_df = comparison_df.sort_values('F1 Score', ascending=False)
print(comparison_df.to_string(index=False))

# Select best model
BEST_MODEL_NAME = comparison_df.iloc[0]['Model']
print(f'\nBest model: {BEST_MODEL_NAME}')

---
## 9. Save Models & Artifacts

In [ ]:
# Save all models
detection_dir = MODELS_DIR / 'threat_detection'
detection_dir.mkdir(parents=True, exist_ok=True)

models_to_save = {
    'cnn_threat_detector': (model_cnn, trainer_cnn),
    'lstm_threat_detector': (model_lstm, trainer_lstm),
    'hybrid_threat_detector': (model_hybrid, trainer_hybrid),
}

for name, (model, trainer) in models_to_save.items():
    # Save model state dict
    model_path = detection_dir / f'{name}.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_class': model.__class__.__name__,
        'n_features': N_FEATURES,
        'training_history': trainer.history,
        'dataset': DATASET_NAME,
        'timestamp': datetime.now().isoformat(),
    }, model_path)
    print(f'Saved: {model_path}')

# Save best model separately for easy loading
best_model_map = {'CNN': model_cnn, 'LSTM': model_lstm, 'Hybrid': model_hybrid}
best_model = best_model_map[BEST_MODEL_NAME]
best_path = detection_dir / 'best_threat_detector.pth'
torch.save({
    'model_state_dict': best_model.state_dict(),
    'model_class': best_model.__class__.__name__,
    'n_features': N_FEATURES,
    'best_model_name': BEST_MODEL_NAME,
    'dataset': DATASET_NAME,
    'metrics': {k: {m: results[k][m] for m in ['accuracy', 'f1', 'auc', 'mcc']} for k in results},
    'timestamp': datetime.now().isoformat(),
}, best_path)
print(f'\nBest model saved: {best_path}')

# Save comparison results
comparison_df.to_csv(detection_dir / 'model_comparison.csv', index=False)
print(f'Comparison saved: {detection_dir / "model_comparison.csv"}')

In [ ]:
# Inference helper function for downstream notebooks
def load_threat_detector(model_path: str, device: str = 'cpu'):
    """
    Load a trained threat detection model for inference.
    Usage in other notebooks:
        model, metadata = load_threat_detector('../models/threat_detection/best_threat_detector.pth')
        model.eval()
        with torch.no_grad():
            probs = torch.softmax(model(X_tensor), dim=1)[:, 1]
    """
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    n_features = checkpoint['n_features']
    model_class = checkpoint['model_class']
    
    # Instantiate correct model class
    if model_class == 'CNNThreatDetector':
        model = CNNThreatDetector(n_features)
    elif model_class == 'LSTMThreatDetector':
        model = LSTMThreatDetector(n_features)
    elif model_class == 'HybridCNNLSTMDetector':
        model = HybridCNNLSTMDetector(n_features)
    else:
        raise ValueError(f'Unknown model class: {model_class}')
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    return model, checkpoint

# Quick test
test_model, test_meta = load_threat_detector(
    str(best_path), device=str(DEVICE)
)
with torch.no_grad():
    sample = torch.FloatTensor(X_test[:5]).to(DEVICE)
    probs = torch.softmax(test_model(sample), dim=1)
    print(f'\nInference test (first 5 samples):')
    print(f'  Predictions: {probs.argmax(dim=1).cpu().numpy()}')
    print(f'  True labels: {y_test[:5]}')
    print(f'  Attack probs: {probs[:, 1].cpu().numpy().round(4)}')

print('\n✓ Notebook 02 complete. Ready for Notebook 03 (Anomaly Detection).')

In [ ]:
# Cleanup
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Memory freed.')